# Anotar posições das imagens de `Data/jiujitsu_images_out_of_sample/`

Passa por cada imagem, mostra as duas poses do ViTPose (**pose1 em vermelho com o número 1**,
**pose2 em azul com o número 2**) e pede para você escolher a `position` num menu.

Cada clique em **Salvar e próxima** grava na hora em `Data/jiujitsu_images_out_of_sample_annotations.json`,
um array de registros no formato:

```json
{"image": "mount/image_0001.jpg", "frame": 41, "position": "mount1",
 "pose1": [[x, y, conf], ... 17 pontos], "pose2": [[x, y, conf], ... 17 pontos]}
```

- `image`: caminho relativo a `Data/jiujitsu_images_out_of_sample/` (`pasta_imagem/nome_imagem`).
- `frame`: número de ordem da imagem na lista (1, 2, 3...), já que as imagens não vêm de vídeo.
- `pose1`/`pose2`: ficam de fora do registro se o detector não achou aquela pessoa.

Se você parar e rodar de novo, ele continua da primeira imagem ainda não anotada.
Kernel: **JiujitsuCompVision (.venv)**. Rode as células em ordem.

In [ ]:
import hashlib
import io
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "experiments").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PASTA = ROOT / "Data" / "jiujitsu_images_out_of_sample"
SAIDA = ROOT / "Data" / "jiujitsu_images_out_of_sample_annotations.json"
CACHE_POSES = ROOT / "diagnostics" / "poses" / "cache_por_hash.json"  # sha256 da imagem -> poses
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

POSITIONS = [
    "5050_guard", "back1", "back2", "closed_guard1", "closed_guard2",
    "half_guard1", "half_guard2", "mount1", "mount2", "open_guard1", "open_guard2",
    "side_control1", "side_control2", "standing", "takedown1", "takedown2",
    "turtle1", "turtle2",
]

imagens = sorted(p for p in PASTA.rglob("*")
                 if p.is_file() and p.suffix.lower() in EXTS and ".ipynb_checkpoints" not in p.parts)
print(f"{len(imagens)} imagens em {PASTA}")

## Poses (ViTPose)

As poses ficam num cache indexado pelo **conteúdo** da imagem (sha256), então renomear ou
apagar arquivos não troca as poses de lugar. O detector só é carregado se faltar alguma pose.

In [ ]:
_detector = None
_cache = json.loads(CACHE_POSES.read_text()) if CACHE_POSES.is_file() else {}


def _sha256(caminho):
    return hashlib.sha256(Path(caminho).read_bytes()).hexdigest()


def _salvar_cache():
    CACHE_POSES.parent.mkdir(parents=True, exist_ok=True)
    tmp = CACHE_POSES.with_suffix(".tmp")
    tmp.write_text(json.dumps(_cache))
    tmp.replace(CACHE_POSES)


def obter_poses(caminho):
    """Devolve (pose1, pose2); cada uma é 17 x [x, y, conf] ou None."""
    global _detector
    chave = _sha256(caminho)
    if chave not in _cache:
        if _detector is None:
            import torch
            from experiments.compare_pose_detectors import VitPoseDetector
            try:
                _detector = VitPoseDetector("cuda" if torch.cuda.is_available() else "cpu")
            except torch.OutOfMemoryError:
                print("GPU sem memória livre (outro kernel usando?). Usando CPU, que é mais lenta.")
                torch.cuda.empty_cache()
                _detector = VitPoseDetector("cpu")
        poses = [[[float(v) for v in ponto] for ponto in pose] for pose in _detector(caminho)]
        _cache[chave] = poses
        _salvar_cache()
    poses = _cache[chave]
    return (poses[0] if len(poses) > 0 else None,
            poses[1] if len(poses) > 1 else None)

### Opcional: extrair tudo antes de anotar
Assim a navegação fica instantânea. Se pular esta célula, cada pose é extraída quando a imagem aparece.

In [ ]:
for i, caminho in enumerate(imagens, 1):
    p1, p2 = obter_poses(caminho)
    n = (p1 is not None) + (p2 is not None)
    print(f"[{i}/{len(imagens)}] {caminho.relative_to(PASTA)}: {n} pessoa(s)")

## Desenho: pose1 em vermelho com "1", pose2 em azul com "2"

In [ ]:
from PIL import Image, ImageDraw, ImageFont

VERMELHO, AZUL = "#ff2d20", "#1f6bff"
SKELETON = [(15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6),
            (5, 7), (6, 8), (7, 9), (8, 10), (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)]


def desenhar(caminho, pose1, pose2, min_score=0.3):
    with Image.open(caminho) as fonte:
        imagem = fonte.convert("RGB")
    tela = ImageDraw.Draw(imagem)
    lado = max(imagem.size)
    r = max(2, round(lado / 250))
    fonte_num = ImageFont.load_default(size=max(18, round(lado / 14)))
    for pose, cor, rotulo in ((pose1, VERMELHO, "1"), (pose2, AZUL, "2")):
        if pose is None:
            continue
        visiveis = [p for p in pose if p[2] >= min_score] or pose
        for a, b in SKELETON:
            if pose[a][2] >= min_score and pose[b][2] >= min_score:
                tela.line([tuple(pose[a][:2]), tuple(pose[b][:2])], fill=cor, width=r)
        for x, y, s in pose:
            if s >= min_score:
                tela.ellipse([x - r, y - r, x + r, y + r], fill=cor)
        # Número ao lado do ponto mais alto da pose, dentro da imagem.
        x, y, _ = min(visiveis, key=lambda p: p[1])
        caixa = tela.textbbox((0, 0), rotulo, font=fonte_num, stroke_width=r)
        w, h = caixa[2] - caixa[0], caixa[3] - caixa[1]
        tx = min(max(x + 2 * r, 0), imagem.width - w)
        ty = min(max(y - h - 2 * r, 0), imagem.height - h)
        tela.text((tx, ty), rotulo, fill=cor, font=fonte_num, stroke_width=r, stroke_fill="white")
    return imagem

## Anotar
Escolha a posição no menu e clique em **Salvar e próxima**. **Anterior** volta (mostrando o que você já anotou),
**Pular** avança sem salvar.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

anotacoes = {}
if SAIDA.is_file():
    anotacoes = {r["image"]: r for r in json.loads(SAIDA.read_text())}
print(f"{len(anotacoes)} imagens já anotadas em {SAIDA.name}")


def salvar_arquivo():
    ordem = {str(p.relative_to(PASTA)): i for i, p in enumerate(imagens)}
    registros = sorted(anotacoes.values(), key=lambda r: ordem.get(r["image"], len(ordem)))
    tmp = SAIDA.with_suffix(".tmp")
    tmp.write_text(json.dumps(registros, indent=1))
    tmp.replace(SAIDA)


rels = [str(p.relative_to(PASTA)) for p in imagens]
estado = {"i": next((i for i, rel in enumerate(rels) if rel not in anotacoes), 0)}

menu = widgets.Dropdown(options=[("— escolha a posição —", None)] + [(p, p) for p in POSITIONS],
                        description="position:", layout=widgets.Layout(width="320px"))
b_ant = widgets.Button(description="◀ Anterior")
b_pular = widgets.Button(description="Pular ▶")
b_salvar = widgets.Button(description="Salvar e próxima ▶", button_style="success", disabled=True)
info = widgets.HTML()
tela = widgets.Image(format="jpeg", layout=widgets.Layout(max_width="900px", max_height="650px"))


def mostrar():
    i = estado["i"]
    if i >= len(imagens):
        info.value = f"<b>Fim!</b> {len(anotacoes)}/{len(imagens)} anotadas em <code>{SAIDA}</code>"
        tela.value = b""
        return
    caminho, rel = imagens[i], rels[i]
    pose1, pose2 = obter_poses(caminho)
    buf = io.BytesIO()
    desenhar(caminho, pose1, pose2).save(buf, format="JPEG", quality=90)
    tela.value = buf.getvalue()
    feito = anotacoes.get(rel)
    menu.value = feito["position"] if feito else None
    faltando = [n for n, p in (("pose1", pose1), ("pose2", pose2)) if p is None]
    aviso = f" — <span style='color:#c00'>sem {', '.join(faltando)}</span>" if faltando else ""
    marca = f" — já anotada como <b>{feito['position']}</b>" if feito else ""
    info.value = (f"<b>[{i + 1}/{len(imagens)}]</b> <code>{rel}</code> (pasta: <b>{Path(rel).parent}</b>)"
                  f"{marca}{aviso} | anotadas: {len(anotacoes)}/{len(imagens)}"
                  f"<br><span style='color:{VERMELHO}'><b>1 = pose1 (vermelho)</b></span> · "
                  f"<span style='color:{AZUL}'><b>2 = pose2 (azul)</b></span>")


def ao_mudar_menu(_):
    b_salvar.disabled = menu.value is None or estado["i"] >= len(imagens)


def ao_salvar(_):
    i = estado["i"]
    if menu.value is None or i >= len(imagens):
        return
    pose1, pose2 = obter_poses(imagens[i])
    registro = {"image": rels[i], "frame": i + 1, "position": menu.value}
    if pose1 is not None:
        registro["pose1"] = pose1
    if pose2 is not None:
        registro["pose2"] = pose2
    anotacoes[rels[i]] = registro
    salvar_arquivo()
    estado["i"] += 1
    mostrar()


def ao_pular(_):
    estado["i"] = min(estado["i"] + 1, len(imagens))
    mostrar()


def ao_voltar(_):
    estado["i"] = max(estado["i"] - 1, 0)
    mostrar()


menu.observe(ao_mudar_menu, names="value")
b_salvar.on_click(ao_salvar)
b_pular.on_click(ao_pular)
b_ant.on_click(ao_voltar)

display(widgets.VBox([info, widgets.HBox([menu, b_ant, b_pular, b_salvar]), tela]))
mostrar()

## Conferir o arquivo salvo

In [ ]:
registros = json.loads(SAIDA.read_text()) if SAIDA.is_file() else []
print(f"{len(registros)} registros em {SAIDA}")
if registros:
    exemplo = dict(registros[0])
    for k in ("pose1", "pose2"):
        if k in exemplo:
            exemplo[k] = f"{len(exemplo[k])} pontos, ex.: {[round(v, 2) for v in exemplo[k][0]]}"
    print(json.dumps(exemplo, indent=1))